In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

tickers = ['AAPL', 'MSFT', 'GOOG']

end_date = datetime.now()

start_date = end_date - timedelta(days=5*365)

print(f"Selected Tickers: {tickers}")
print(f"Data will be fetched from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")

data = yf.download(tickers, start=start_date, end=end_date)
df = data['Close']

print("Shape of the acquired data:", df.shape)
print("First 5 rows of the acquired data:")
print(df.head())

print("Number of missing values per ticker:\n", df.isnull().sum())

df_filled = df.ffill()

print("\nNumber of missing values after forward fill:\n", df_filled.isnull().sum())

df_cleaned = df_filled.dropna()

print("\nShape of data after handling missing values:", df_cleaned.shape)
print("First 5 rows of cleaned data:")
print(df_cleaned.head())

from sklearn.preprocessing import MinMaxScaler

train_size = int(len(df_cleaned) * 0.8)
train_data = df_cleaned.iloc[:train_size]
test_data = df_cleaned.iloc[train_size:]

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train_data = scaler.fit_transform(train_data)
scaled_test_data = scaler.transform(test_data)
scaled_train_df = pd.DataFrame(scaled_train_data, columns=df_cleaned.columns, index=train_data.index)
scaled_test_df = pd.DataFrame(scaled_test_data, columns=df_cleaned.columns, index=test_data.index)

print("\nFirst 5 rows of scaled training data:")
print(scaled_train_df.head())
print("\nFirst 5 rows of scaled testing data:")
print(scaled_test_df.head())

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 7))
for column in scaled_train_df.columns:
    plt.plot(scaled_train_df.index, scaled_train_df[column], label=column)

plt.title('Scaled Training Data Over Time')
plt.xlabel('Date')
plt.ylabel('Scaled Close Price')
plt.legend()
plt.grid(True)
plt.show()

differenced_train_df = scaled_train_df.diff().dropna()
differenced_test_df = scaled_test_df.diff().dropna()

print("\nShape of differenced training data:", differenced_train_df.shape)
print("First 5 rows of differenced training data:")
print(differenced_train_df.head())

print("\nShape of differenced testing data:", differenced_test_df.shape)
print("First 5 rows of differenced testing data:")
print(differenced_test_df.head())

import numpy as np

def create_sequences(data, look_back):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:(i + look_back)])
        y.append(data[i + look_back])
    return np.array(X), np.array(y)

look_back = 60  

differenced_train_array = differenced_train_df.values
differenced_test_array = differenced_test_df.values

X_train, y_train = create_sequences(differenced_train_array, look_back)
X_test, y_test = create_sequences(differenced_test_array, look_back)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nFirst sample of X_train (look_back window):")
print(X_train[0])
print("\nFirst sample of y_train (target value):")
print(y_train[0])

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

n_features = X_train.shape[2]

model = Sequential()
model.add(LSTM(units=50, activation='relu', return_sequences=True, input_shape=(look_back, n_features)))
model.add(LSTM(units=50, activation='relu'))
model.add(Dense(units=n_features))
model.compile(optimizer='adam', loss='mse')

model.summary()
print("LSTM model architecture defined and compiled.")

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1)

print("\nModel training complete.")

loss = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss (MSE): {loss:.4f}")

from sklearn.metrics import mean_absolute_error, mean_squared_error

y_pred = model.predict(X_test)
scaled_test_data_undifferenced_ref = scaled_test_df.values[look_back-1:-1]

if scaled_test_data_undifferenced_ref.shape == y_pred.shape:
    scaled_predictions = y_pred + scaled_test_data_undifferenced_ref
    scaled_true_values = y_test + scaled_test_data_undifferenced_ref
else:
    print("Shape mismatch during inverse differencing reference selection.")
    print(f"y_pred shape: {y_pred.shape}")
    print(f"scaled_test_data_undifferenced_ref shape: {scaled_test_data_undifferenced_ref.shape}")
    min_len = min(len(y_pred), len(scaled_test_data_undifferenced_ref))
    scaled_predictions = y_pred[:min_len] + scaled_test_data_undifferenced_ref[:min_len]
    scaled_true_values = y_test[:min_len] + scaled_test_data_undifferenced_ref[:min_len]
    print(f"Adjusted shapes to: {scaled_predictions.shape}")

original_predictions = scaler.inverse_transform(scaled_predictions)

original_true_values = scaler.inverse_transform(scaled_true_values)

print("Predictions and true values inverse transformed to original scale.")

mae = mean_absolute_error(original_true_values, original_predictions)
mse = mean_squared_error(original_true_values, original_predictions)
rmse = np.sqrt(mse)

print(f"\nMean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("\nFirst 5 original predictions:")
print(original_predictions[:5])
print("\nFirst 5 original true values:")
print(original_true_values[:5])

import matplotlib.pyplot as plt

plt.figure(figsize=(16, 8))
plt.plot(original_true_values[:, 0], label='Actual AAPL Prices')
plt.plot(original_predictions[:, 0], label='Predicted AAPL Prices')
plt.title('AAPL Stock Price Prediction vs. Actuals')
plt.xlabel('Time Step')
plt.ylabel('Stock Price')
plt.legend()
plt.grid(True)
plt.show()

print("Plot of actual vs. predicted prices for AAPL displayed.")

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

arima_predictions = {}

for ticker in df_cleaned.columns:
    train_series = scaled_train_df[ticker]
    model = ARIMA(train_series, order=(5,1,0))
    model_fit = model.fit()
    forecast_result = model_fit.forecast(steps=len(scaled_test_df))
    arima_predictions[ticker] = forecast_result.values 

scaled_arima_predictions_df = pd.DataFrame(arima_predictions, index=scaled_test_df.index, columns=df_cleaned.columns)

print("Scaled ARIMA predictions generated:")
print(scaled_arima_predictions_df.head())

original_arima_predictions = scaler.inverse_transform(scaled_arima_predictions_df)
first_true_value_date = differenced_test_df.index[look_back]
original_arima_predictions_df = pd.DataFrame(original_arima_predictions, columns=df_cleaned.columns, index=scaled_test_df.index)
arima_predictions_aligned = original_arima_predictions_df.loc[first_true_value_date:].values
min_len_eval = min(len(original_true_values), len(arima_predictions_aligned))
original_true_values_aligned = original_true_values[:min_len_eval]
arima_predictions_aligned = arima_predictions_aligned[:min_len_eval]

print(f"\nShape of original_true_values_aligned: {original_true_values_aligned.shape}")
print(f"Shape of arima_predictions_aligned: {arima_predictions_aligned.shape}")

mae_arima = mean_absolute_error(original_true_values_aligned, arima_predictions_aligned)
mse_arima = mean_squared_error(original_true_values_aligned, arima_predictions_aligned)
rmse_arima = np.sqrt(mse_arima)

print(f"\nARIMA Model Performance:")
print(f"Mean Absolute Error (MAE): {mae_arima:.4f}")
print(f"Mean Squared Error (MSE): {mse_arima:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_arima:.4f}")

print("First 5 original ARIMA predictions (aligned):")
print(arima_predictions_aligned[:5])

print("The ARIMA model has been implemented, trained, and evaluated for each stock ticker. The performance metrics (MAE, MSE, RMSE) have been calculated and printed, providing a baseline for comparison with the deep learning model.")

from statsmodels.tsa.api import ExponentialSmoothing

ets_predictions = {}

for ticker in df_cleaned.columns:
    train_series = scaled_train_df[ticker]
    model_ets = ExponentialSmoothing(train_series, trend='add', seasonal='add', seasonal_periods=20, initialization_method="estimated")
    model_fit_ets = model_ets.fit()
    forecast_result_ets = model_fit_ets.forecast(steps=len(scaled_test_df))
    ets_predictions[ticker] = forecast_result_ets.values

scaled_ets_predictions_df = pd.DataFrame(ets_predictions, index=scaled_test_df.index, columns=df_cleaned.columns)

print("Scaled Exponential Smoothing predictions generated:")
print(scaled_ets_predictions_df.head())

original_ets_predictions = scaler.inverse_transform(scaled_ets_predictions_df)
first_true_value_date = differenced_test_df.index[look_back]

original_ets_predictions_df = pd.DataFrame(original_ets_predictions, columns=df_cleaned.columns, index=scaled_test_df.index)

ets_predictions_aligned = original_ets_predictions_df.loc[first_true_value_date:].values

min_len_eval_ets = min(len(original_true_values_aligned), len(ets_predictions_aligned))
ets_predictions_aligned = ets_predictions_aligned[:min_len_eval_ets]

print(f"\nShape of original_true_values_aligned: {original_true_values_aligned.shape}")
print(f"Shape of ets_predictions_aligned: {ets_predictions_aligned.shape}")

mae_ets = mean_absolute_error(original_true_values_aligned, ets_predictions_aligned)
mse_ets = mean_squared_error(original_true_values_aligned, ets_predictions_aligned)
rmse_ets = np.sqrt(mse_ets)

print(f"\nExponential Smoothing Model Performance:")
print(f"Mean Absolute Error (MAE): {mae_ets:.4f}")
print(f"Mean Squared Error (MSE): {mse_ets:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ets:.4f}")

print("First 5 original Exponential Smoothing predictions (aligned):")
print(ets_predictions_aligned[:5])

print("The Exponential Smoothing model has been implemented, trained, and evaluated for each stock ticker. The performance metrics (MAE, MSE, RMSE) have been calculated and printed, providing another baseline for comparison with the deep learning model.")

from prophet import Prophet
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

prophet_predictions = {}

for ticker in df_cleaned.columns:
    prophet_train_df = scaled_train_df[[ticker]].reset_index()
    prophet_train_df.columns = ['ds', 'y']
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)

    model_prophet = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
    model_prophet.fit(prophet_train_df)

    future = model_prophet.make_future_dataframe(periods=len(scaled_test_df), include_history=False)

    forecast = model_prophet.predict(future)

    prophet_predictions[ticker] = forecast['yhat'].values

scaled_prophet_predictions_df = pd.DataFrame(prophet_predictions, index=scaled_test_df.index, columns=df_cleaned.columns)

print("Scaled Prophet predictions generated:")
print(scaled_prophet_predictions_df.head())

original_prophet_predictions = scaler.inverse_transform(scaled_prophet_predictions_df)

first_true_value_date = differenced_test_df.index[look_back]

original_prophet_predictions_df = pd.DataFrame(original_prophet_predictions, columns=df_cleaned.columns, index=scaled_test_df.index)

prophet_predictions_aligned = original_prophet_predictions_df.loc[first_true_value_date:].values

min_len_eval_prophet = min(len(original_true_values_aligned), len(prophet_predictions_aligned))
prophet_predictions_aligned = prophet_predictions_aligned[:min_len_eval_prophet]

print(f"\nShape of original_true_values_aligned: {original_true_values_aligned.shape}")
print(f"Shape of prophet_predictions_aligned: {prophet_predictions_aligned.shape}")

mae_prophet = mean_absolute_error(original_true_values_aligned, prophet_predictions_aligned)
mse_prophet = mean_squared_error(original_true_values_aligned, prophet_predictions_aligned)
rmse_prophet = np.sqrt(mse_prophet)

print(f"\nProphet Model Performance:")
print(f"Mean Absolute Error (MAE): {mae_prophet:.4f}")
print(f"Mean Squared Error (MSE): {mse_prophet:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_prophet:.4f}")

print("First 5 original Prophet predictions (aligned):")
print(prophet_predictions_aligned[:5])

import shap
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(units=50, activation='relu', return_sequences=True, input_shape=(look_back, n_features)))
model.add(LSTM(units=50, activation='relu'))
model.add(Dense(units=n_features))
model.compile(optimizer='adam', loss='mse')

print("Keras LSTM model re-instantiated (untrained) to resolve variable overwrite issue.")

if X_train.shape[0] > 100:
    idx = np.random.choice(X_train.shape[0], 100, replace=False)
    X_background = X_train[idx]
else:
    X_background = X_train

print(f"Shape of X_background: {X_background.shape}")

def predict_wrapper(data):
    num_samples = data.shape[0]
    reshaped_data = data.reshape(num_samples, look_back, n_features)
    return model.predict(reshaped_data, verbose=0)

explainer = shap.KernelExplainer(predict_wrapper, X_background.reshape(X_background.shape[0], -1))

print("SHAP KernelExplainer initialized.")

import shap
import numpy as np

X_test_subset = X_test[:5]

print(f"Shape of X_test_subset: {X_test_subset.shape}")
X_test_subset_flattened = X_test_subset.reshape(X_test_subset.shape[0], -1)
shap_values = explainer.shap_values(X_test_subset_flattened)

print(f"Number of output features for SHAP values: {len(shap_values)}")
print(f"Shape of SHAP values for the first output: {shap_values[0].shape}")
print("SHAP values computed successfully.")

import matplotlib.pyplot as plt

feature_names = []
for t in range(look_back):
    for i, col in enumerate(df_cleaned.columns):
        feature_names.append(f'{col}_t-{look_back-1-t}')

shap_values_for_sample_0_output_0 = shap_values[0][:, 0]

expected_value_for_output_0 = explainer.expected_value[0]

shap.plots.waterfall(shap.Explanation(
    values=shap_values_for_sample_0_output_0,
    base_values=expected_value_for_output_0,
    data=X_test_subset_flattened[0],
    feature_names=feature_names
), max_display=20)

plt.title('SHAP Waterfall Plot for AAPL First Prediction')
plt.tight_layout()
plt.show()

print("SHAP waterfall plot displayed for the first prediction of AAPL.")

import pandas as pd

metrics_data = {
    'LSTM': {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    },
    'ARIMA': {
        'MAE': mae_arima,
        'MSE': mse_arima,
        'RMSE': rmse_arima
    },
    'Exponential Smoothing': {
        'MAE': mae_ets,
        'MSE': mse_ets,
        'RMSE': rmse_ets
    },
    'Prophet': {
        'MAE': mae_prophet,
        'MSE': mse_prophet,
        'RMSE': rmse_prophet
    }
}

metrics_df = pd.DataFrame(metrics_data).T 

print("Comparative Performance Metrics:")
print(metrics_df)

import matplotlib.pyplot as plt

ticker_index = 0
ticker_name = df_cleaned.columns[ticker_index]

plt.figure(figsize=(18, 9))

plt.plot(original_true_values_aligned[:, ticker_index], label=f'Actual {ticker_name} Prices', color='black', linewidth=2)

plt.plot(original_predictions[:, ticker_index], label=f'LSTM Predicted {ticker_name} Prices', color='blue', linestyle='--')

plt.plot(arima_predictions_aligned[:, ticker_index], label=f'ARIMA Predicted {ticker_name} Prices', color='green', linestyle='-.')

plt.plot(ets_predictions_aligned[:, ticker_index], label=f'ETS Predicted {ticker_name} Prices', color='red', linestyle=':')

plt.plot(prophet_predictions_aligned[:, ticker_index], label=f'Prophet Predicted {ticker_name} Prices', color='purple', linestyle=':')

plt.title(f'{ticker_name} Stock Price: Actual vs. Predictions from all Models', fontsize=16)
plt.xlabel('Time Step (Test Data)', fontsize=12)
plt.ylabel('Stock Price', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Plot of actual vs. predicted prices for {ticker_name} from all models displayed.")

import matplotlib.pyplot as plt
import numpy as np

model_names = metrics_df.index
mae_scores = metrics_df['MAE']
mse_scores = metrics_df['MSE']
rmse_scores = metrics_df['RMSE']

x = np.arange(len(model_names)) 
width = 0.2 

fig, ax = plt.subplots(figsize=(15, 8))

rects1 = ax.bar(x - width, mae_scores, width, label='MAE', color='skyblue')
rects2 = ax.bar(x, mse_scores, width, label='MSE', color='lightcoral')
rects3 = ax.bar(x + width, rmse_scores, width, label='RMSE', color='lightgreen')

ax.set_ylabel('Error Score', fontsize=12)
ax.set_title('Comparative Performance Metrics Across Models', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha="right")
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.7)

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

fig.tight_layout()
plt.show()

print("Bar chart comparing MAE, MSE, and RMSE metrics across all models displayed.")
